# Final Analysis and Insights
This notebook provides final analysis of the pain medication effectiveness prediction model, including patient trait analysis, condition-specific insights, example predictions, and key findings.

## 1. Import Required Libraries

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## 2. Load Saved Model and Test Data

In [ ]:
# TODO: Load saved model and related files
model_path = '../../outputs/models/rf_model.pkl'
feature_names_path = '../../outputs/models/feature_names.pkl'
feature_importance_path = '../../outputs/models/feature_importance.csv'
test_predictions_path = '../../outputs/models/test_predictions.csv'
processed_data_path = '../../data/processed/pain_meds_ml_ready.csv'

# Load model
with open(model_path, 'rb') as f:
    rf_model = pickle.load(f)

# Load feature names
with open(feature_names_path, 'rb') as f:
    feature_names = pickle.load(f)

# Load feature importance
feature_importance = pd.read_csv(feature_importance_path)

# Load test predictions
test_results = pd.read_csv(test_predictions_path)

# Load original processed data
df = pd.read_csv(processed_data_path)

print("All files loaded successfully!")
print(f"Model: {type(rf_model).__name__}")
print(f"Number of features: {len(feature_names)}")
print(f"Test predictions: {len(test_results)} samples")
print(f"Dataset: {df.shape}")

## 3. Analyze Which Patient Traits Matter Most

In [ ]:
# TODO: Analyze which features/traits are most predictive
print("=" * 60)
print("PATIENT TRAIT ANALYSIS")
print("=" * 60)

# Group features by type
drug_features = feature_importance[feature_importance['feature'].str.startswith('drug_')]
condition_features = feature_importance[feature_importance['feature'].str.startswith('condition_')]
text_features = feature_importance[
    feature_importance['feature'].isin(['review_length', 'review_word_count', 
                                        'has_positive_keywords', 'has_negative_keywords', 
                                        'avg_word_length'])
]
other_features = feature_importance[
    ~feature_importance['feature'].str.startswith('drug_') & 
    ~feature_importance['feature'].str.startswith('condition_') &
    ~feature_importance['feature'].isin(text_features['feature'])
]

print("\nFeature Type Analysis:")
print(f"Drug-related features: {len(drug_features)} (Total importance: {drug_features['importance'].sum():.4f})")
print(f"Condition-related features: {len(condition_features)} (Total importance: {condition_features['importance'].sum():.4f})")
print(f"Text-based features: {len(text_features)} (Total importance: {text_features['importance'].sum():.4f})")
print(f"Other features: {len(other_features)} (Total importance: {other_features['importance'].sum():.4f})")

# Top drugs by importance
print("\n" + "=" * 60)
print("TOP 10 MOST PREDICTIVE DRUGS:")
print("=" * 60)
display(drug_features.head(10))

# Top conditions by importance
print("\n" + "=" * 60)
print("TOP 10 MOST PREDICTIVE CONDITIONS:")
print("=" * 60)
display(condition_features.head(10))

# Text features
if len(text_features) > 0:
    print("\n" + "=" * 60)
    print("TEXT FEATURE IMPORTANCE:")
    print("=" * 60)
    display(text_features.sort_values('importance', ascending=False))

# Visualize feature type importance
feature_type_importance = pd.DataFrame({
    'Feature Type': ['Drugs', 'Conditions', 'Text Features', 'Other'],
    'Total Importance': [
        drug_features['importance'].sum(),
        condition_features['importance'].sum(),
        text_features['importance'].sum() if len(text_features) > 0 else 0,
        other_features['importance'].sum()
    ]
})

plt.figure(figsize=(10, 6))
plt.bar(feature_type_importance['Feature Type'], feature_type_importance['Total Importance'], 
        color=['coral', 'lightgreen', 'skyblue', 'lightgray'])
plt.ylabel('Total Importance Score', fontsize=12)
plt.title('Feature Type Importance Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. Identify Which Conditions Respond Best

In [ ]:
# TODO: Analyze effectiveness by condition
print("=" * 60)
print("CONDITION-SPECIFIC EFFECTIVENESS ANALYSIS")
print("=" * 60)

if 'condition' in df.columns and 'effectiveness' in df.columns:
    # Calculate average rating by condition
    condition_stats = df.groupby('condition').agg({
        'rating': ['mean', 'count'],
        'effectiveness': lambda x: (x == 'Effective').sum() / len(x) * 100
    }).round(2)
    
    condition_stats.columns = ['avg_rating', 'count', 'effectiveness_rate']
    condition_stats = condition_stats[condition_stats['count'] >= 50]  # Filter for conditions with at least 50 reviews
    condition_stats = condition_stats.sort_values('effectiveness_rate', ascending=False)
    
    print("\nTop 15 Conditions with Best Response (min 50 reviews):")
    display(condition_stats.head(15))
    
    print("\nBottom 15 Conditions with Poorest Response (min 50 reviews):")
    display(condition_stats.tail(15))
    
    # Visualize top and bottom conditions
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Top 10 conditions
    top_10 = condition_stats.head(10)
    axes[0].barh(range(len(top_10)), top_10['effectiveness_rate'], color='green', alpha=0.7)
    axes[0].set_yticks(range(len(top_10)))
    axes[0].set_yticklabels(top_10.index)
    axes[0].set_xlabel('Effectiveness Rate (%)', fontsize=11)
    axes[0].set_title('Top 10 Conditions - Best Response', fontsize=13, fontweight='bold')
    axes[0].invert_yaxis()
    
    # Bottom 10 conditions
    bottom_10 = condition_stats.tail(10)
    axes[1].barh(range(len(bottom_10)), bottom_10['effectiveness_rate'], color='red', alpha=0.7)
    axes[1].set_yticks(range(len(bottom_10)))
    axes[1].set_yticklabels(bottom_10.index)
    axes[1].set_xlabel('Effectiveness Rate (%)', fontsize=11)
    axes[1].set_title('Bottom 10 Conditions - Poorest Response', fontsize=13, fontweight='bold')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
else:
    print("Condition or effectiveness columns not found in dataset.")

## 5. Create Example Predictions with Confidence Scores

In [ ]:
# TODO: Create 5-10 example predictions with detailed breakdown
print("=" * 60)
print("EXAMPLE PREDICTIONS WITH CONFIDENCE SCORES")
print("=" * 60)

# Get mapping for effectiveness labels
effectiveness_labels = {
    0: 'Not Effective',
    1: 'Partially Effective',
    2: 'Effective'
}

# Select 10 diverse examples from test set
np.random.seed(42)
sample_indices = np.random.choice(len(test_results), size=10, replace=False)

examples = []
for i, idx in enumerate(sample_indices, 1):
    actual = test_results.iloc[idx]['actual']
    predicted = test_results.iloc[idx]['predicted']
    prob_not_eff = test_results.iloc[idx]['prob_not_effective']
    prob_partial = test_results.iloc[idx]['prob_partially_effective']
    prob_eff = test_results.iloc[idx]['prob_effective']
    
    # Get highest probability
    max_prob = max(prob_not_eff, prob_partial, prob_eff)
    confidence = max_prob * 100
    
    # Determine if prediction is correct
    correct = "✓" if actual == predicted else "✗"
    
    example = {
        'Example': i,
        'Actual': effectiveness_labels[int(actual)],
        'Predicted': effectiveness_labels[int(predicted)],
        'Confidence': f"{confidence:.1f}%",
        'Correct': correct,
        'Prob_Not_Eff': f"{prob_not_eff*100:.1f}%",
        'Prob_Partial': f"{prob_partial*100:.1f}%",
        'Prob_Effective': f"{prob_eff*100:.1f}%"
    }
    examples.append(example)
    
    print(f"\nExample {i}:")
    print(f"  Actual: {effectiveness_labels[int(actual)]}")
    print(f"  Predicted: {effectiveness_labels[int(predicted)]} ({correct})")
    print(f"  Confidence: {confidence:.1f}%")
    print(f"  Probability Breakdown:")
    print(f"    - Not Effective: {prob_not_eff*100:.1f}%")
    print(f"    - Partially Effective: {prob_partial*100:.1f}%")
    print(f"    - Effective: {prob_eff*100:.1f}%")

# Create summary dataframe
examples_df = pd.DataFrame(examples)
print("\n" + "=" * 60)
print("SUMMARY TABLE:")
print("=" * 60)
display(examples_df)

## 6. Key Findings Summary

### TODO: Summarize key findings from the entire analysis

## Key Findings

### 1. Model Performance
- **Overall Accuracy:** [Insert test accuracy from notebook 05]
- **Precision/Recall/F1:** [Summarize from classification report]
- **Model Strengths:** [Which classes does it predict best?]
- **Model Weaknesses:** [Which classes does it struggle with?]

### 2. Most Important Predictive Factors
- **Top Drug Features:** [List top 3-5 drugs that strongly predict effectiveness]
- **Top Condition Features:** [List top 3-5 conditions that strongly predict effectiveness]
- **Text Features Impact:** [How much do review characteristics matter?]
- **Surprising Findings:** [Any unexpected feature importances?]

### 3. Condition-Specific Insights
- **Best Responding Conditions:** [Which pain conditions show highest effectiveness rates?]
- **Poorest Responding Conditions:** [Which pain conditions show lowest effectiveness rates?]
- **Clinical Implications:** [What might explain these differences?]

### 4. Drug Effectiveness Patterns
- **Most Effective Medications:** [Which drugs consistently receive high ratings?]
- **Least Effective Medications:** [Which drugs consistently receive low ratings?]
- **Drug Class Patterns:** [Any patterns by drug type - NSAIDs vs opioids vs acetaminophen?]

### 5. Prediction Confidence
- **High Confidence Predictions:** [What percentage of predictions have >80% confidence?]
- **Low Confidence Predictions:** [When does the model struggle to decide?]
- **Confidence vs Accuracy:** [Do high confidence predictions tend to be more accurate?]

### 6. Practical Applications
- **Clinical Use Cases:** [How could this model be used in practice?]
- **Patient Counseling:** [How could results inform patient discussions?]
- **Treatment Selection:** [How could this guide medication choices?]

### 7. Limitations
- **Data Limitations:** [What biases or gaps exist in the data?]
- **Model Limitations:** [What are the model's blind spots?]
- **Generalization Concerns:** [How well would this work on different populations?]

### 8. Recommendations for Future Work
- **Model Improvements:** [What could improve model performance?]
- **Additional Features:** [What data would be valuable to include?]
- **Alternative Approaches:** [Should other models be explored?]

### 9. Business/Clinical Value
- **Cost Savings:** [Potential for reduced trial-and-error prescribing?]
- **Patient Outcomes:** [Potential for improved pain management?]
- **Implementation Considerations:** [What would deployment require?]

## 7. Export Results for Final Report

In [ ]:
# TODO: Export key tables and figures for final report
output_dir = '../../outputs/final_results/'
os.makedirs(output_dir, exist_ok=True)

# Export feature importance
feature_importance.head(20).to_csv(output_dir + 'top_features.csv', index=False)
print(f"Exported: {output_dir}top_features.csv")

# Export drug importance
drug_features.head(15).to_csv(output_dir + 'top_drugs.csv', index=False)
print(f"Exported: {output_dir}top_drugs.csv")

# Export condition importance
condition_features.head(15).to_csv(output_dir + 'top_conditions.csv', index=False)
print(f"Exported: {output_dir}top_conditions.csv")

# Export condition effectiveness analysis
if 'condition_stats' in locals():
    condition_stats.to_csv(output_dir + 'condition_effectiveness.csv')
    print(f"Exported: {output_dir}condition_effectiveness.csv")

# Export example predictions
examples_df.to_csv(output_dir + 'example_predictions.csv', index=False)
print(f"Exported: {output_dir}example_predictions.csv")

# Create summary statistics file
summary_stats = {
    'Metric': [
        'Total Features',
        'Drug Features',
        'Condition Features',
        'Text Features',
        'Test Samples',
        'Correct Predictions',
        'Accuracy Rate'
    ],
    'Value': [
        len(feature_names),
        len(drug_features),
        len(condition_features),
        len(text_features),
        len(test_results),
        (test_results['actual'] == test_results['predicted']).sum(),
        f"{((test_results['actual'] == test_results['predicted']).sum() / len(test_results) * 100):.2f}%"
    ]
}
summary_df = pd.DataFrame(summary_stats)
summary_df.to_csv(output_dir + 'summary_statistics.csv', index=False)
print(f"Exported: {output_dir}summary_statistics.csv")

print("\n" + "=" * 60)
print("FINAL ANALYSIS COMPLETE")
print("=" * 60)
print(f"All results exported to: {output_dir}")
print("\nFiles created:")
print("  - top_features.csv")
print("  - top_drugs.csv")
print("  - top_conditions.csv")
print("  - condition_effectiveness.csv")
print("  - example_predictions.csv")
print("  - summary_statistics.csv")
print("=" * 60)